# Gradio lab: a web interface for the support chatbot

**Problem.** The food-delivery support assistant from Module 3 lives in a notebook. Only someone who can run Python can try it. Before the company builds anything bigger (Units 4.1–4.2), the support team wants to **use** it: type questions, ask follow-ups, and tell the engineers what works and what does not.

**Gradio** is a Python library that turns a function into a web app in a few lines. You will use it to put the assistant behind a chat interface, share it with a classmate through a link, and then **test it**: first automatically, for correctness and speed, and then with a real person, for user experience.

**Learning objectives**

1. Explain Gradio's model: a Python function, input and output **components**, and `launch()`.
2. Build a `gr.Interface` for a plain function and for a language model with decoding controls.
3. Build a chatbot with `gr.ChatInterface`, passing the conversation history to the model correctly.
4. Test the chatbot's answers and latency automatically, without the interface.
5. Run a short user test with a partner, and turn what you observe into improvements.
6. Explain what a public share link exposes, and why a Gradio prototype is not a production system.

**Prerequisites:** [RAG with speech](../../../module3/lectures/rag/rag_speech.ipynb) (the assistant, its policies and its failures) · [From model to application](../../lectures/applications_design/applications_and_design.md) (Units 4.1–4.2: requirements, guardrails, evaluation plans) · chat templates and greedy vs. sampled decoding (Modules 2–3).

**Data and model**

- **Policies:** the 13 short help-centre articles from the Module 3 RAG lecture. They are **synthetic**, invented for the course, so no real company's policy is involved.
- **Model:** [Qwen/Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct) on a GPU (about 3.1 GB download), or [Qwen/Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct) on a CPU (about 1 GB). Both are Apache-2.0. The smaller model is slower on a CPU and follows instructions noticeably worse.
- **No retrieval.** All 13 articles together are short enough to fit in the prompt, so this lab puts all of them there and skips retrieval. The lab is about the interface. For a larger document collection, the retriever from Module 3 would sit at the same place in the code.

**Environment and compute**

- **Kaggle notebook, GPU T4, internet on.** Internet is needed for the downloads **and** for the share link. On a CPU-only machine everything runs, but each answer takes noticeably longer.
- Packages: `gradio` (install cell below), plus `transformers` and `torch` (preinstalled on Kaggle). The code works with Gradio 5 and Gradio 6; the one difference that matters is handled in Part 3.
- No training. Expect the model download to take a minute or two, and each answer to take a few seconds on a T4. Plan about **two hours** in total, including the partner test.

**Files.** The chat log from Part 5 is written to `/kaggle/working/gradio_lab/chat_log.csv` (or `./gradio_lab_outputs/chat_log.csv` elsewhere). Rerunning Part 5 adds rows to it; delete the file to start again.

**How to work:** This version contains completed code and saved automated checks. Run cells in order, or stop after a launch cell to use that app locally. The final cleanup cell closes the last app. Automated browser observations are clearly labelled; the human partner study is not claimed as completed.

**Marks (20):** Ex. 1 — 2 · Ex. 2 — 3 · Ex. 3 — 4 · Ex. 4 — 4 · Ex. 5 — 5 · reflection — 2.


## Setup

**Installation (Kaggle).** Run the next cell once per session, it takes about a minute. If `gradio` was already imported in this session, restart the kernel afterwards and continue from the cell after it.

In [1]:
import importlib.util
if importlib.util.find_spec('gradio') is None:
    get_ipython().run_line_magic('pip', 'install -q "gradio> =5,<7"')
import gradio
print('Gradio available:', gradio.__version__)


Note: you may need to restart the kernel to use updated packages.


 /home/akash/.conda/envs/nitin_clean/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Gradio available: 6.31.0


In [2]:
import csv
import json
import os
import re
import statistics
import time
from datetime import datetime
from pathlib import Path
from threading import Thread

import gradio as gr
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer

torch.manual_seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GRADIO_MAJOR = int(gr.__version__.split('.')[0])
OUT = Path("/kaggle/working/gradio_lab") if Path("/kaggle/working").is_dir() else Path("gradio_lab_outputs")
OUT.mkdir(parents=True, exist_ok=True)
SHARE = Path("/kaggle/working").is_dir() or bool(os.environ.get("COLAB_RELEASE_TAG"))
# Local runs use localhost; set SHARE=True explicitly if you need a public link.

print('gradio', gr.__version__, '| transformers', transformers.__version__, '| torch', torch.__version__)
print('device:', DEVICE, '| outputs:', OUT)

def show_browser_evidence(stage):
    """Display saved automated browser evidence, when the verification runner is used."""
    path = OUT / f"browser_{stage}.json"
    if path.exists():
        report = json.loads(path.read_text())
        print('Saved automated browser evidence (the browser checks are run by the verification runner):')
        print(json.dumps(report, indent=2, ensure_ascii=False))
        return report
    print('No automated browser report for this run. Use the interface for the manual checks.')
    return None


gradio 6.31.0 | transformers 5.20.2 | torch 2.14.1+cu130
device: cuda | outputs: gradio_lab_outputs


**Opening the app.** On this local machine, open the printed `http://127.0.0.1:7860` address. Local execution uses `SHARE=False`, so a public tunnel is not needed. On Kaggle or Colab, `SHARE` defaults to `True`; you can also explicitly enable it to request a temporary public link. A tunnel failure does not prevent a local app from running. Anyone with a working public link can use the app and its compute while the session is alive.

The helper below closes the previous app before starting the next one. For interactive testing, run one section at a time; the cleanup cell at the end closes the final app.

In [3]:
_running = []

def launch(demo):
    """Keep one app running; local execution does not require a public tunnel."""
    while _running:
        _running.pop().close()
    demo.launch(share=SHARE, prevent_thread_lock=True)
    _running.append(demo)


## Part 1 — Gradio in ten minutes

A Gradio app has three parts:

```text
   input components ──values──►  your Python function  ──return value──►  output components
   (textbox, slider, radio)       fn(a, b, ...)                            (textbox, label, image)
                         └──────────────── launch(): a web page that calls fn on every submit ─────────────┘
```

1. **A function.** Ordinary Python. Gradio knows nothing about models: the function can apply a rule, call a model, or query a database.
2. **Components.** Each input component supplies one argument to the function, **in order**. Each output component displays one return value. Commonly used components:

| Component | Shows | Value passed to / from the function |
| --- | --- | --- |
| `gr.Textbox` | A text field | `str` |
| `gr.Number` | A number field | `float` (or `int` with `precision=0`) |
| `gr.Slider` | A slider with `minimum`, `maximum`, `step` | `float` |
| `gr.Radio`, `gr.Dropdown` | A fixed list of `choices` | the chosen `str` |
| `gr.Checkbox` | A tick box | `bool` |
| `gr.Image`, `gr.Audio` | Media upload, microphone, or display | a NumPy array or a file path |

3. **`launch()`.** Starts a web server. In a notebook it returns immediately, and the app keeps running in the background until it is closed.

`gr.Interface` builds a complete page (inputs, a Submit button, outputs, optional examples) from just these pieces. Here it wraps the **late-delivery rule** from the policies. This is a rule, not a model: as Unit 4.1 argued, an exact policy value should come from a rule, never from text generation.

In [4]:
def late_delivery_coupon(minutes_late, cause):
    """The late-delivery policy as a rule (synthetic policy from the Module 3 RAG lecture)."""
    if cause != "None of these":
        return f"No coupon: delays caused by {cause.lower()} do not qualify."
    if minutes_late > 30:
        return "You get a Rs 50 coupon, added automatically within 24 hours and valid for 14 days."
    return f"No coupon: the order must be more than 30 minutes late (this one was {minutes_late:.0f} minutes late)."

demo_rule = gr.Interface(
    fn=late_delivery_coupon,
    inputs=[gr.Slider(0, 120, value=40, step=1, label="Minutes after the promised time"),
            gr.Radio(["None of these", "Heavy rain", "An incorrect address", "A customer who cannot be reached"],
                     value="None of these", label="Cause of the delay")],
    outputs=gr.Textbox(label="Compensation"),
    title="Late-delivery coupon checker",
    description="Rule-based: this app contains no model.",
    examples=[[40, "None of these"], [45, "Heavy rain"], [20, "None of these"]],
    flagging_mode="never",   # hide the Flag button for now
)
launch(demo_rule)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
